# M3 — Government QA RAG + LoRA

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
# !pip installs (safe to re-run)
!pip install -U transformers peft accelerate pandas pyarrow tqdm --quiet

In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

In [ ]:
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
# ===== User config (EDIT ME) =====
BASE_MODEL   = "mistralai/Mistral-7B-Instruct-v0.3"
LORA_DIR     = "/Volumes/main/default/thesis_project/M3/Govern_M3_Test/test_out_M3_Lite_GovReport_1.0"  # ← 你的 M3 LoRA（test）目录
TEST_JSONL   = "/Volumes/main/default/thesis_project/M3/Test_GovReport_e5_1.0/govqa_test_topk.jsonl"  # ← test 的 top-k jsonl

# Outputs
OUT_CSV      = "/Volumes/main/default/thesis_project/M3/Govern_M3_Test/pred_govqa_test_M3_frozen_1.0.csv"
OUT_PARQUET  = "/Volumes/main/default/thesis_project/M3/Govern_M3_Test/pred_govqa_test_M3_frozen_1.0.parquet"

# Generation defaults（测试建议可复现）
DO_SAMPLE        = False
MAX_NEW_TOKENS   = 224
SEED             = 42

# Context controls
PASSAGE_CHAR_LIMIT = 800     # 每段截断上限（为 True 冻结且“不截断”请看 FREEZE_NO_TRIM_TEXT）
FREEZE_TOPK_JSONL   = True   # 冻结 JSONL：不 rerank，只按原顺序取
RERANK_TOPM         = 6      # 冻结模式下，仅“取前 M 条”；≤0 则取全部
FREEZE_NO_TRIM_TEXT = False  # True 则不对 passage 做字符截断（最严格冻结，但上下文可能过长）

# Citation scoring（轻量）
USE_E5_SIM   = True          # 用 e5-small-v2 做句子-段落相似度（更稳）；False 仅用词重合 F1
E5_MODEL_ID  = "intfloat/e5-small-v2"
MIN_CITE_SCORE = 0.06        # 句子附引的最低相似阈值
MAX_TOTAL_CITES = 2          # 单答案中最多不同来源标注数（可设 None/0 不限）


In [ ]:
# ============================
# Setup
# ============================
import os, gc, json, re, shutil, hashlib, numpy as np, pandas as pd, torch, random
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig

# Repro & CUDA hygiene
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
gc.collect(); torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================
# Utilities
# ============================
def ensure_dir(p: str): Path(p).mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

# --- LoRA loader with config sanitize (PEFT-version-safe) ---
ALLOWED_KEYS = {
    "r","lora_alpha","lora_dropout","target_modules","fan_in_fan_out","bias",
    "use_dora","use_rslora","init_lora_weights","inference_mode",
    "rank_pattern","alpha_pattern","modules_to_save","layers_to_transform","layers_pattern",
    "task_type","peft_type","auto_mapping","base_model_name_or_path","revision",
    "loftq_config","megablocks"
}

In [ ]:
def sanitize_and_load_lora(base_model_obj, adapter_dir: str) -> PeftModel:
    cfg_path = os.path.join(adapter_dir, "adapter_config.json")
    assert os.path.exists(cfg_path), f"adapter_config.json is missing in: {adapter_dir}"
    with open(cfg_path, "r") as f: cfg = json.load(f)
    try: shutil.copy(cfg_path, cfg_path + ".bak_autofix")
    except Exception: pass

    # Flatten nested blocks; mark DoRA if present
    for nested_key in ("lora_config", "dora_config", "corda_config"):
        if nested_key in cfg and isinstance(cfg[nested_key], dict):
            for k, v in cfg.pop(nested_key).items():
                cfg.setdefault(k, v)
            if nested_key in ("dora_config", "corda_config"):
                cfg["use_dora"] = True

    # Drop unknown *_config (keep loftq_config)
    for k in list(cfg.keys()):
        if re.search(r"_config$", k) and k not in ("loftq_config",):
            cfg.pop(k, None)

    cfg["peft_type"] = "LORA"
    cfg.setdefault("task_type", "CAUSAL_LM")
    if not cfg.get("target_modules"):
        cfg["target_modules"] = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

    clean = {k: v for k, v in cfg.items() if k in ALLOWED_KEYS}
    print("Using cleaned LoRA config keys:", sorted(clean.keys()))
    peft_conf = LoraConfig(**clean)
    model = PeftModel.from_pretrained(base_model_obj, adapter_dir, config=peft_conf)
    return model.eval()

In [ ]:
# ============================
# Load model + tokenizer + LoRA
# ============================
# Tokenizer: prefer LoRA dir to preserve exact chat template
tok = AutoTokenizer.from_pretrained(LORA_DIR if os.path.isdir(LORA_DIR) else BASE_MODEL, use_fast=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 推理阶段与训练对齐：左截断，左填充（能更好保留尾部结构）
tok.truncation_side = "left"
tok.padding_side    = "left"

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=dtype, device_map=None, low_cpu_mem_usage=True
).to("cuda" if torch.cuda.is_available() else "cpu").eval()

model = sanitize_and_load_lora(base, LORA_DIR).to(base.device).eval()
print("✓ LoRA adapter loaded:", LORA_DIR)


In [ ]:
!pip install faiss-cpu --quiet
import faiss

In [ ]:
# =========================================
# Minimal RAG builders for TEST top-k JSONL
# (drop-in: defines build_doc_map_from_json_or_evidence,
#           build_global_index, build_topk_jsonl)
# =========================================

# !pip install -U transformers faiss-cpu pandas tqdm --quiet

import os, json, re, gc
from pathlib import Path
from typing import Any, Dict, List, Tuple
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch

# ----------------
# Config (edit if needed)
# ----------------
EMB_MODEL      = "intfloat/e5-large-v2"
EMB_DIM        = 1024
EMB_MAXLEN     = 512

CHUNK_SIZE     = 1500
CHUNK_OVERLAP  = 200
TOPK           = 8
OVERFETCH      = 64              # 先召回 N，再过滤
RESTRICT_SAME_DOC = True        # 只保留同 report_id 的段落
ALLOW_EVIDENCE_FALLBACK = True  # JSON 不在时，是否用 oracle_evidence 作为回退

# 你已有的目录（按你的环境设置）
# GOVROOT = "/Volumes/main/default/thesis_project/GovernmentDocument/gov-report"
# CRS_DIR = os.path.join(GOVROOT, "crs")
# GAO_DIR = os.path.join(GOVROOT, "gao")

# 你已有的工具：load_parquet / extract_qas（如没有，这里提供极简版）
def load_parquet(path: str) -> pd.DataFrame:
    req = ['report_id','question','gold_answer','qid']
    df = pd.read_parquet(path)
    miss = [c for c in req if c not in df.columns]
    if miss:
        raise ValueError(f"{path} missing cols: {miss}")
    return df

def extract_qas(df: pd.DataFrame) -> List[Dict]:
    out = []
    for i, r in df.iterrows():
        out.append({
            "qid": str(r.get("qid", f"row_{i:06d}")),
            "doc_id": str(r["report_id"]),
            "question": str(r["question"]).strip(),
            "gold_answer": str(r["gold_answer"]).strip()
        })
    return out

# ----------------
# Helpers: read & flatten CRS/GAO JSON
# ----------------
def parse_oracle_evidence(x: Any) -> List[str]:
    if x is None: return []
    if isinstance(x, str):
        s = x.strip()
        if not s: return []
        try: obj = json.loads(s)
        except: return [s]
    else:
        obj = x
    out = []
    if isinstance(obj, list):
        for e in obj:
            if isinstance(e, str) and e.strip(): out.append(e.strip())
            elif isinstance(e, dict):
                t = e.get("text") or e.get("paragraph") or ""
                if isinstance(t, str) and t.strip(): out.append(t.strip())
    elif isinstance(obj, dict):
        ps = obj.get("paragraphs", [])
        for p in ps:
            if isinstance(p, str) and p.strip(): out.append(p.strip())
            elif isinstance(p, dict):
                t = p.get("text","")
                if isinstance(t, str) and t.strip(): out.append(t.strip())
    return out

def _read_json_if_exists(path: str):
    if path and os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return json.load(f)
        except:
            return None
    return None

def _id_candidates(report_id: str) -> List[str]:
    rid = str(report_id).strip()
    cands = [rid, rid.replace("/", "_"), rid.replace(" ", "_"), rid.replace(":", "_")]
    cands += [c.lower() for c in list(cands)]
    seen, out = set(), []
    for c in cands:
        if c not in seen:
            seen.add(c); out.append(c)
    return out

def _yield_paragraphs_from_node(node: Any, level: int = 0, skip_letter_top_paras: bool = True):
    if node is None: return
    if isinstance(node, str):
        t = node.strip()
        if t: yield t
        return
    if isinstance(node, list):
        for x in node:
            yield from _yield_paragraphs_from_node(x, level=level, skip_letter_top_paras=skip_letter_top_paras)
        return
    if not isinstance(node, dict): return

    title = None
    for k in ["title", "heading", "section_name", "name"]:
        if isinstance(node.get(k), str) and node[k].strip():
            title = node[k].strip(); break
    is_letter_top = (level == 0 and title and title.lower() == "letter")

    paras = None
    for k in ["paragraphs", "paras", "para_list"]:
        if isinstance(node.get(k), list):
            paras = node[k]; break
    if paras is None and isinstance(node.get("text"), str):
        paras = [node["text"]]

    if paras and not (skip_letter_top_paras and is_letter_top):
        for p in paras:
            if isinstance(p, str):
                t = p.strip()
                if t: yield t
            elif isinstance(p, dict):
                txt = p.get("text", "")
                if isinstance(txt, str) and txt.strip():
                    yield txt.strip()

    for ck in ["subsections", "children", "sections", "parts", "items"]:
        if isinstance(node.get(ck), list):
            for ch in node[ck]:
                yield from _yield_paragraphs_from_node(ch, level=level+1, skip_letter_top_paras=skip_letter_top_paras)

def flatten_report_json(obj: Dict, is_gao: bool) -> str:
    parts: List[str] = []
    if not obj or not isinstance(obj, dict): return ""
    if not is_gao:
        summ = obj.get("summary")
        if summ: parts.extend([s for s in _yield_paragraphs_from_node(summ) if s])
    else:
        hi = obj.get("highlight"); ff = obj.get("fastfact")
        if hi: parts.extend([s for s in _yield_paragraphs_from_node(hi) if s])
        if ff: parts.extend([s for s in _yield_paragraphs_from_node(ff) if s])
    rep = obj.get("report")
    if rep: parts.extend([s for s in _yield_paragraphs_from_node(rep, level=0, skip_letter_top_paras=is_gao) if s])
    return "\n\n".join(parts)

def load_report_fulltext(report_id: str, crs_dir: str, gao_dir: str) -> str:
    for rid in _id_candidates(report_id):
        p_crs = os.path.join(crs_dir, f"{rid}.json")
        obj = _read_json_if_exists(p_crs)
        if obj is not None: return flatten_report_json(obj, is_gao=False)
        p_gao = os.path.join(gao_dir, f"{rid}.json")
        obj = _read_json_if_exists(p_gao)
        if obj is not None: return flatten_report_json(obj, is_gao=True)
    return ""

# ----------------
# Embeddings (e5)
# ----------------
from transformers import AutoTokenizer as HFAutoTokenizer, AutoModel as HFAutoModel
e5_tok = HFAutoTokenizer.from_pretrained(EMB_MODEL)
e5_enc = HFAutoModel.from_pretrained(EMB_MODEL)

def chunk_text(text: str, size: int, overlap: int) -> List[str]:
    if not text: return []
    step = max(1, size - overlap)
    out, i, L = [], 0, len(text)
    while i < L:
        ch = text[i:i+size]
        out.append(ch)
        if len(ch) < size: break
        i += step
    return out

@torch.no_grad()
def encode_passages(texts: List[str], bs: int = 32, device: str = None) -> np.ndarray:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    enc = e5_enc.to(device).eval()
    vecs = []
    for i in tqdm(range(0, len(texts), bs), desc="Encode [passage]"):
        batch = ["passage: " + t for t in texts[i:i+bs]]
        toks  = e5_tok(batch, padding=True, truncation=True, max_length=EMB_MAXLEN, return_tensors="pt").to(device)
        h     = enc(**toks).last_hidden_state[:, 0]
        h     = torch.nn.functional.normalize(h, p=2, dim=1)
        vecs.append(h.cpu().numpy().astype("float32"))
    return np.vstack(vecs) if vecs else np.zeros((0, EMB_DIM), dtype="float32")

@torch.no_grad()
def encode_queries(texts: List[str], bs: int = 64, device: str = None) -> np.ndarray:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    enc = e5_enc.to(device).eval()
    vecs = []
    for i in tqdm(range(0, len(texts), bs), desc="Encode [query]"):
        batch = ["query: " + t for t in texts[i:i+bs]]
        toks  = e5_tok(batch, padding=True, truncation=True, max_length=EMB_MAXLEN, return_tensors="pt").to(device)
        h     = enc(**toks).last_hidden_state[:, 0]
        h     = torch.nn.functional.normalize(h, p=2, dim=1)
        vecs.append(h.cpu().numpy().astype("float32"))
    return np.vstack(vecs) if vecs else np.zeros((0, EMB_DIM), dtype="float32")

# ----------------
# FAISS
# ----------------
import faiss

def build_doc_map_from_json_or_evidence(df: pd.DataFrame,
                                        crs_dir: str, gao_dir: str,
                                        allow_evidence_fallback: bool = True) -> Dict[str, str]:
    by_report = {}
    miss_json, used_fallback = 0, 0
    for rid, g in tqdm(df.groupby("report_id"), desc="Collect fulltexts by report_id"):
        rid = str(rid)
        full = load_report_fulltext(rid, crs_dir, gao_dir)
        if not full.strip():
            miss_json += 1
            if allow_evidence_fallback and "oracle_evidence" in g.columns:
                evs = []
                for e in g["oracle_evidence"].tolist():
                    evs.extend(parse_oracle_evidence(e))
                evs = [x for x in (e.strip() for e in evs) if x]
                evs = list(dict.fromkeys(evs))
                full = "\n\n".join(evs)
                if full.strip(): used_fallback += 1
        if full.strip():
            by_report[rid] = full
    print(f"Built doc map: {len(by_report)} documents. Missing JSON: {miss_json} | used evidence fallback: {used_fallback}")
    return by_report

def build_global_index(doc_map: Dict[str, str], split: str, out_dir_index: str) -> Tuple[str, str]:
    print(f"\n=== Building FAISS for split: {split} ===")
    texts, pids, docids = [], [], []
    for did, full in tqdm(doc_map.items(), desc=f"Chunking {split}"):
        chunks = chunk_text(full.strip(), CHUNK_SIZE, CHUNK_OVERLAP)
        for j, ch in enumerate(chunks):
            texts.append(ch)
            pids.append(f"{did}_p{j:04d}")
            docids.append(did)

    X = encode_passages(texts, bs=32)
    assert X.shape[1] == EMB_DIM, f"Embedding dim mismatch: {X.shape}"

    index = faiss.IndexFlatIP(EMB_DIM)
    index.add(X)

    split_dir = Path(out_dir_index) / split
    split_dir.mkdir(parents=True, exist_ok=True)
    idx_path  = str(split_dir / f"govqa_{split}_e5.index")
    meta_path = str(split_dir / f"govqa_{split}_meta.jsonl")

    faiss.write_index(index, idx_path)
    with open(meta_path, "w", encoding="utf-8") as f:
        for did, pid, txt in zip(docids, pids, texts):
            f.write(json.dumps({"doc_id": did, "pid": pid, "text": txt}, ensure_ascii=False) + "\n")

    print(f"✓ {split}: ntotal={index.ntotal} -> {idx_path}\n  meta -> {meta_path}")
    return idx_path, meta_path

def build_topk_jsonl(qas: List[Dict], idx_path: str, meta_path: str, out_jsonl: str):
    print(f"\n=== Build Top-K JSONL → {out_jsonl} ===")
    index = faiss.read_index(idx_path)
    meta  = [json.loads(l) for l in open(meta_path, "r", encoding="utf-8")]
    corpus_texts = [m["text"] for m in meta]
    corpus_docid = [m["doc_id"] for m in meta]

    Q = encode_queries([x["question"] for x in qas], bs=64)

    with open(out_jsonl, "w", encoding="utf-8") as f:
        for x, qv in tqdm(zip(qas, Q), total=len(qas), desc="Search + filter"):
            did = x["doc_id"]
            D, I = index.search(qv.reshape(1, -1), OVERFETCH)
            hits = []
            for j in I[0]:
                if j < 0: continue
                if RESTRICT_SAME_DOC and corpus_docid[j] != did:
                    continue
                hits.append({"pid": f"{corpus_docid[j]}_p{j}", "text": corpus_texts[j]})
                if len(hits) >= TOPK:
                    break
            if not hits:
                # 兜底：允许跨文档（保持管线不报错）
                hits = [{"pid": f"{corpus_docid[j]}_p{j}", "text": corpus_texts[j]}
                        for j in I[0][:TOPK] if j >= 0]
            rec = {**x, "topk_passages": hits}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"✓ wrote {out_jsonl}")


In [ ]:
import os

# 根目录（按你的实际路径确认）
GOVROOT = "/Volumes/main/default/thesis_project/GovernmentDocument/gov-report"
CRS_DIR = os.path.join(GOVROOT, "crs")
GAO_DIR = os.path.join(GOVROOT, "gao")

print("CRS_DIR:", CRS_DIR)
print("GAO_DIR:", GAO_DIR)

In [ ]:
# ============================
# Build TEST split Top-K JSONL
# ============================
from pathlib import Path

TEST_QA_PARQUET = "/Volumes/main/default/thesis_project/GovernmentDocument/gov-report-qs/processed_20250804_234209/qs_test_qa_evidence.parquet"
OUT_DIR_INDEX   = "/Volumes/main/default/thesis_project/M3/Govern_M3_Test/test_GovReport_e5_1.0"
OUT_TEST_JSONL  = str(Path(OUT_DIR_INDEX) / "govqa_test_topk.jsonl")

Path(OUT_DIR_INDEX).mkdir(parents=True, exist_ok=True)

print("Reading TEST parquet ...")
df_test = load_parquet(TEST_QA_PARQUET)
print("TEST rows:", len(df_test))

print("Collecting fulltexts for TEST ...")
doc_map_test = build_doc_map_from_json_or_evidence(
    df_test, CRS_DIR, GAO_DIR, allow_evidence_fallback=ALLOW_EVIDENCE_FALLBACK
)

test_idx, test_meta = build_global_index(doc_map_test, "test", OUT_DIR_INDEX)

qas_test = extract_qas(df_test)
print("QAs for TEST:", len(qas_test))

build_topk_jsonl(qas_test, test_idx, test_meta, OUT_TEST_JSONL)
print("✓ Test Top-K JSONL ready at:", OUT_TEST_JSONL)

# 供后续推理脚本直接使用
TEST_JSONL = OUT_TEST_JSONL

In [ ]:
# ============================
# Frozen Top-K JSONL: hash + size
# ============================
assert os.path.exists(TEST_JSONL), f"TEST_JSONL not found: {TEST_JSONL}"
j_hash = sha256_file(TEST_JSONL)
j_size = os.path.getsize(TEST_JSONL)
print(f"[FREEZE] TEST_JSONL sha256={j_hash} | size={j_size} bytes")

In [ ]:
# ============================
# Inference helpers (no rerank; keep order)
# ============================
@torch.no_grad()
def decode_new_tokens(model, tok, prompt: str, *, max_new_tokens: int, do_sample: bool):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    in_len = inputs["input_ids"].shape[1]
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        repetition_penalty=1.05,
        no_repeat_ngram_size=6,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
    )
    if do_sample:
        gen_kwargs.update(dict(temperature=0.4, top_p=0.9))
    out = model.generate(**inputs, **gen_kwargs)
    new_tokens = out[0, in_len:]
    text = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return text

def build_ctx_frozen(passages, char_limit=PASSAGE_CHAR_LIMIT, no_trim=False):
    def _c(t):
        t = (t or "").replace("\n"," ").strip()
        if no_trim or not char_limit:
            return t
        return (t[:char_limit] + " ...") if len(t) > char_limit else t
    return "\n".join(f"[s{i+1}] {_c(p.get('text',''))}" for i, p in enumerate(passages))

@torch.no_grad()
def generate_answer_only_frozen(model, tok, passages, question,
                                max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE):
    ctx = build_ctx_frozen(passages, PASSAGE_CHAR_LIMIT, no_trim=FREEZE_NO_TRIM_TEXT)
    SYS = (
        "You are a careful research assistant. Use ONLY the provided sources.\n"
        "Write ONE or TWO concise sentences that explicitly state the factual answer "
        "(avoid yes/no or bare lists). If unknown, say \"I don't know.\" "
        "Do not repeat the system or the context."
    )
    msgs = [
        {"role":"system","content":SYS},
        {"role":"user","content":f"[CONTEXT]\n{ctx}\n[/CONTEXT]\n\nQuestion: {question}"},
    ]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    ans = decode_new_tokens(model, tok, prompt, max_new_tokens=max_new_tokens, do_sample=do_sample)

    # clean residues
    ans = re.sub(r"<<SYS>>.*?<</SYS>>", "", ans, flags=re.S)
    ans = re.sub(r"\[CONTEXT\].*?\[/CONTEXT\]", "", ans, flags=re.S)
    ans = re.sub(r"\bBIBREF\d+\b", "", ans).strip()

    # avoid too-short/yes-no (only if sampling allowed)
    def is_too_short_or_yesno(a: str):
        w = a.split()
        if len(w) < 6: return True
        low = a.lower().strip()
        return low in {"yes", "no"} or low.startswith(("yes", "no"))
    if is_too_short_or_yesno(ans) and DO_SAMPLE:
        msgs2 = msgs + [
            {"role":"assistant","content":ans},
            {"role":"user","content":"Rewrite as ONE factual sentence (no yes/no), include the key term/value explicitly."}
        ]
        prompt2 = tok.apply_chat_template(msgs2, add_generation_prompt=True, tokenize=False)
        ans2 = decode_new_tokens(model, tok, prompt2, max_new_tokens=max_new_tokens, do_sample=True)
        ans2 = re.sub(r"<<SYS>>.*?<</SYS>>", "", ans2, flags=re.S)
        ans2 = re.sub(r"\[CONTEXT\].*?\[/CONTEXT\]", "", ans2, flags=re.S)
        ans2 = re.sub(r"\bBIBREF\d+\b", "", ans2).strip()
        if len(ans2.split()) >= len(ans.split()):
            ans = ans2

    # 120-word cap
    words = ans.split()
    if len(words) > 120:
        ans = " ".join(words[:120])
    return ans

In [ ]:
# ============================
# Sentence-wise citation (order-preserving)
# ============================
_e5_tok = _e5_enc = None

def ensure_e5():
    global _e5_tok, _e5_enc
    if _e5_tok is None or _e5_enc is None:
        from transformers import AutoTokenizer, AutoModel
        dev = model.device
        _e5_tok = AutoTokenizer.from_pretrained(E5_MODEL_ID)
        _e5_enc = AutoModel.from_pretrained(E5_MODEL_ID).to(dev).eval()

def norm_words(s):
    return re.findall(r"[a-zA-Z0-9]+", (s or "").lower())

def overlap_f1(a, b):
    aw, bw = norm_words(a), norm_words(b)
    if not aw or not bw: return 0.0
    aset, bset = set(aw), set(bw)
    inter = len(aset & bset)
    prec = inter / max(1, len(aset))
    rec  = inter / max(1, len(bset))
    return 0.0 if prec+rec == 0 else 2*prec*rec/(prec+rec)

@torch.no_grad()
def e5_cos(a, b):
    ensure_e5()
    t = _e5_tok([f"query: {a}", f"passage: {b}"], padding=True, truncation=True,
                max_length=256, return_tensors="pt").to(_e5_enc.device)
    h = _e5_enc(**t).last_hidden_state[:,0]
    h = torch.nn.functional.normalize(h, p=2, dim=1)
    return float((h[0] @ h[1]).item())  # [-1,1]

def fused_score(q, p, w_sem=0.6):
    f1 = overlap_f1(q, p)
    if USE_E5_SIM:
        sem = e5_cos(q, p)
        sem01 = (sem + 1) / 2  # → [0,1]
        return (1 - w_sem) * f1 + w_sem * sem01
    return f1

def sent_split(text: str):
    parts = re.split(r'(?<=[\.!?])\s+|\n+', (text or "").strip())
    return [p.strip() for p in parts if p.strip()]

def attach_citations_sentwise(answer, passages, *,
                              per_sent_top=1, min_score_sent=MIN_CITE_SCORE,
                              global_fallback=True, max_total_cites=MAX_TOTAL_CITES, w_sem=0.6):
    """
    不改变 passages 的顺序；只在答案句子末尾添加 [s#]
    """
    sents = sent_split(answer)
    if not sents: return answer

    ptexts = [p.get("text","") or "" for p in passages]
    used = []
    cited_any = False
    out_sents = []

    for s in sents:
        scores = [(fused_score(s, pt, w_sem=w_sem), i+1) for i, pt in enumerate(ptexts)]
        scores.sort(reverse=True)
        picks = [i for sc, i in scores[:per_sent_top] if sc >= min_score_sent]
        if picks:
            cited_any = True
            used.extend(picks)
            out_sents.append(s + " " + " ".join(f"[s{j}]" for j in picks))
        else:
            out_sents.append(s)

    if (not cited_any) and global_fallback and ptexts:
        g_scores = [(fused_score(answer, pt, w_sem=w_sem), i+1) for i, pt in enumerate(ptexts)]
        g_scores.sort(reverse=True)
        if g_scores and g_scores[0][0] >= max(0.05, min_score_sent * 0.7):
            j = g_scores[0][1]
            out_sents[-1] = out_sents[-1] + f" [s{j}]"
            used.append(j)

    if max_total_cites:
        uniq = []
        for j in used:
            if j not in uniq:
                uniq.append(j)
        if len(uniq) > max_total_cites:
            keep = set(uniq[:max_total_cites])
            def filter_tail(sentence):
                return re.sub(r"\[s(\d+)\]", lambda m: m.group(0) if int(m.group(1)) in keep else "", sentence)
            out_sents = [filter_tail(s) for s in out_sents]

    return " ".join(out_sents)

In [ ]:
# ============ Quick preview for frozen TEST JSONL ============
import textwrap, random

def preview_frozen_then_answer(test_jsonl_path, n=5, width=110, topm=RERANK_TOPM):
    items = [json.loads(l) for l in open(test_jsonl_path,"r",encoding="utf-8")]
    items = [s for s in items if s.get("question") and s.get("topk_passages")]
    random.seed(SEED)
    picks = random.sample(items, min(n, len(items)))

    for idx, s in enumerate(picks, 1):
        q = s["question"].strip()
        # 冻结模式：不 rerank，只取前 topm 条
        passages = s["topk_passages"]
        if topm and topm > 0:
            passages = passages[:topm]

        ans = generate_answer_only_frozen(model, tok, passages, q,
                                          max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE)
        ans_cited = attach_citations_sentwise(
            ans, passages, per_sent_top=1, min_score_sent=MIN_CITE_SCORE,
            global_fallback=True, max_total_cites=MAX_TOTAL_CITES, w_sem=0.6
        )

        print("="*width)
        print(f"[{idx}/{len(picks)}] Q:")
        print(textwrap.fill(q, width=width))
        print("-"*width)
        print("Answer (with post-citations):")
        print(textwrap.fill(ans_cited, width=width))
        print("-"*width)
        cites = sorted(set(int(m) for m in re.findall(r"\[s(\d+)\]", ans_cited)))
        if cites:
            print("Cited passages preview:")
            for ci in cites[:2]:
                if 1 <= ci <= len(passages):
                    pv = (passages[ci-1].get("text","") or "").replace("\n"," ").strip()
                    print("•", f"[s{ci}] " + textwrap.shorten(pv, width=width-8, placeholder=" ..."))
        else:
            print("⚠ No [s#] citations detected.")
        print("="*width, "\n")

# Run a quick preview
preview_frozen_then_answer(TEST_JSONL, n=5, topm=RERANK_TOPM)


In [ ]:
# ============================
# MAIN: Inference over TEST (FROZEN Top-K)
# ============================
rows = []
with open(TEST_JSONL, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Infer GovReport TEST (FROZEN top-k)"):
        s = json.loads(line)
        q = (s.get("question") or "").strip()
        topk = s.get("topk_passages", [])
        if not q or not topk:  # 跳过无效样本
            continue

        # —— 冻结：不 rerank；按原顺序取前 M 条（或全部）——
        if FREEZE_TOPK_JSONL:
            passages = topk if (not RERANK_TOPM or RERANK_TOPM <= 0) else topk[:RERANK_TOPM]
        else:
            # 如需非冻结，可改为你的 rerank 逻辑（此处不启用）
            passages = topk[:RERANK_TOPM] if RERANK_TOPM and RERANK_TOPM > 0 else topk

        # 生成答案
        ans = generate_answer_only_frozen(
            model, tok, passages, q,
            max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE
        )

        # 附加引用（不改变 passage 顺序，仅加 [s#]）
        ans_cited = attach_citations_sentwise(
            ans, passages, per_sent_top=1, min_score_sent=MIN_CITE_SCORE,
            global_fallback=True, max_total_cites=MAX_TOTAL_CITES, w_sem=0.6
        )

        # 记录 pid 顺序（若 JSONL 包含 pid）
        pid_list = [(p.get("pid") or "") for p in passages]

        rows.append({
            "qid": s.get("qid",""),
            "doc_id": s.get("doc_id",""),
            "question": q,
            "gold_answer": s.get("gold_answer",""),
            "pid_list": pid_list,
            "contexts": [p.get("text","") for p in passages],
            "answer_M3": ans_cited,
            "topk_sha256": j_hash
        })

In [ ]:
# ============================
# MAIN: Inference over TEST (FROZEN Top-K) — FAST
# ============================
BATCH_SIZE        = 8        # ↑可加大到 16/24 视显存而定
FAST_NO_CITATION  = False    # True 则完全不做句子级引文匹配（最快）
FAST_LEXICAL_CITE = True     # True 用词重合F1替代 e5 语义相似（快很多）
# 若 FAST_NO_CITATION=True，上面这个就忽略了

# 如果选择简化引文，禁用 e5
if FAST_NO_CITATION or FAST_LEXICAL_CITE:
    USE_E5_SIM = False

def build_ctx_frozen_batch(batch_passages, char_limit=PASSAGE_CHAR_LIMIT, no_trim=FREEZE_NO_TRIM_TEXT):
    """把一批样本的 passages 转成 ctx 文本列表（避免在循环里反复做小字符串操作）"""
    out = []
    for passages in batch_passages:
        lines = []
        for i, p in enumerate(passages, 1):
            t = (p.get("text","") or "").replace("\n"," ").strip()
            if not no_trim and char_limit and len(t) > char_limit:
                t = t[:char_limit] + " ..."
            lines.append(f"[s{i+1}] {t}")
        out.append("\n".join(lines))
    return out

@torch.no_grad()
def generate_batch(model, tok, prompts, max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE):
    """一次性生成一批，明显快于逐条"""
    inputs = tok(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    in_lens = [len(x) for x in inputs["input_ids"]]
    gen = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        repetition_penalty=1.05,
        no_repeat_ngram_size=6,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
    )
    outs = []
    for k, in_len in enumerate(in_lens):
        new_tokens = gen[k, in_len:]
        text = tok.decode(new_tokens, skip_special_tokens=True).strip()
        text = re.sub(r"<<SYS>>.*?<</SYS>>", "", text, flags=re.S)
        text = re.sub(r"\[CONTEXT\].*?\[/CONTEXT\]", "", text, flags=re.S)
        text = re.sub(r"\bBIBREF\d+\b", "", text).strip()
        # cap 120 words
        ws = text.split()
        if len(ws) > 120:
            text = " ".join(ws[:120])
        outs.append(text)
    return outs

def make_prompts_from_ctx_questions(ctx_list, questions):
    SYS = (
        "You are a careful research assistant. Use ONLY the provided sources.\n"
        "Write ONE or TWO concise sentences that explicitly state the factual answer "
        "(avoid yes/no or bare lists). If unknown, say \"I don't know.\" "
        "Do not repeat the system or the context."
    )
    prompts = []
    for ctx, q in zip(ctx_list, questions):
        msgs = [
            {"role":"system","content":SYS},
            {"role":"user","content":f"[CONTEXT]\n{ctx}\n[/CONTEXT]\n\nQuestion: {q}"},
        ]
        prompts.append(tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False))
    return prompts

In [ ]:
#耗时过久，改进第二版，先验证
# ============================
# Quick sanity check: run 5 samples (FROZEN, batched)
# ============================
import textwrap

N_PREVIEW = 5
preview_rows = []

# 读取前 N_PREVIEW 个有效样本
valid_samples = []
with open(TEST_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        s = json.loads(line)
        if s.get("question") and s.get("topk_passages"):
            valid_samples.append(s)
        if len(valid_samples) >= N_PREVIEW:
            break

if not valid_samples:
    print("⚠️ No valid samples found in TEST_JSONL for preview.")
else:
    # 1) 冻结：不rerank，只按原顺序取前 M 条（或全部）
    batch_pass_store = []
    batch_q_store = []
    batch_meta_store = []
    for s in valid_samples:
        q = (s.get("question") or "").strip()
        topk = s.get("topk_passages", [])
        passages = topk if (not RERANK_TOPM or RERANK_TOPM <= 0) else topk[:RERANK_TOPM]
        pid_list = [(p.get("pid") or "") for p in passages]

        batch_pass_store.append(passages)
        batch_q_store.append(q)
        batch_meta_store.append((s.get("qid",""), s.get("doc_id",""), s.get("gold_answer",""), pid_list))

    # 2) 批量构造上下文与 prompt
    ctx_list = build_ctx_frozen_batch(batch_pass_store)
    prompts  = make_prompts_from_ctx_questions(ctx_list, batch_q_store)

    # 3) 批量生成
    gen_answers = generate_batch(model, tok, prompts)

    # 4) 句子级引用（依据你的加速开关）
    if FAST_NO_CITATION:
        final_answers = gen_answers
    else:
        final_answers = []
        for ans, passages in zip(gen_answers, batch_pass_store):
            if FAST_LEXICAL_CITE:
                saved_USE_E5 = USE_E5_SIM
                USE_E5_SIM = False
                ans_cited = attach_citations_sentwise(
                    ans, passages, per_sent_top=1, min_score_sent=MIN_CITE_SCORE,
                    global_fallback=True, max_total_cites=MAX_TOTAL_CITES, w_sem=0.0
                )
                USE_E5_SIM = saved_USE_E5
            else:
                ans_cited = attach_citations_sentwise(
                    ans, passages, per_sent_top=1, min_score_sent=MIN_CITE_SCORE,
                    global_fallback=True, max_total_cites=MAX_TOTAL_CITES, w_sem=0.6
                )
            final_answers.append(ans_cited)

    # 5) 打印预览
    width = 110
    for i, ((qid, doc_id, gold, pid_list), q, ctx, ans_cited) in enumerate(
        zip(batch_meta_store, batch_q_store, ctx_list, final_answers), 1
    ):
        print("="*width)
        print(f"[Preview {i}/{len(batch_q_store)}] QID: {qid} | DOC: {doc_id}")
        print(textwrap.fill(q, width=width))
        print("-"*width)
        print("Answer (with citations):")
        print(textwrap.fill(ans_cited, width=width))
        # 简短显示被引用的段落预览
        cites = sorted(set(int(m) for m in re.findall(r"\[s(\d+)\]", ans_cited)))
        if cites:
            print("-"*width)
            print("Cited passages preview:")
            for ci in cites[:2]:
                if 1 <= ci <= len(batch_pass_store[i-1]):
                    pv = (batch_pass_store[i-1][ci-1].get("text","") or "").replace("\n"," ").strip()
                    print("•", f"[s{ci}]", textwrap.shorten(pv, width=width-8, placeholder=" ..."))
        print("="*width, "\n")

In [ ]:
rows = []
batch_pass_store = []   # [[]passages]
batch_q_store    = []   # [question]
batch_meta_store = []   # [(qid, doc_id, gold, pid_list)]

with open(TEST_JSONL, "r", encoding="utf-8") as f:
    it = (json.loads(l) for l in f)
    it = (s for s in it if (s.get("question") and s.get("topk_passages")))
    it = tqdm(it, desc="Infer GovReport TEST (FROZEN top-k, batched)")

    for s in it:
        q = (s.get("question") or "").strip()
        topk = s.get("topk_passages", [])

        # 冻结：不rerank，只按原顺序取前M条（或全部）
        passages = topk if (not RERANK_TOPM or RERANK_TOPM <= 0) else topk[:RERANK_TOPM]
        pid_list = [(p.get("pid") or "") for p in passages]

        batch_pass_store.append(passages)
        batch_q_store.append(q)
        batch_meta_store.append((s.get("qid",""), s.get("doc_id",""), s.get("gold_answer",""), pid_list))

        if len(batch_q_store) >= BATCH_SIZE:
            # 1) 批量构造上下文与 prompt
            ctx_list = build_ctx_frozen_batch(batch_pass_store)
            prompts  = make_prompts_from_ctx_questions(ctx_list, batch_q_store)
            # 2) 批量生成
            gen_answers = generate_batch(model, tok, prompts)

            # 3) 附加引文（可选简化/关闭）
            if FAST_NO_CITATION:
                final_answers = gen_answers
            else:
                final_answers = []
                for ans, passages in zip(gen_answers, batch_pass_store):
                    if FAST_LEXICAL_CITE:   # 只用词重合 F1，避免加载/推理 e5
                        saved_USE_E5 = USE_E5_SIM
                        USE_E5_SIM = False
                        ans_cited = attach_citations_sentwise(
                            ans, passages, per_sent_top=1, min_score_sent=MIN_CITE_SCORE,
                            global_fallback=True, max_total_cites=MAX_TOTAL_CITES, w_sem=0.0
                        )
                        USE_E5_SIM = saved_USE_E5
                    else:
                        ans_cited = attach_citations_sentwise(
                            ans, passages, per_sent_top=1, min_score_sent=MIN_CITE_SCORE,
                            global_fallback=True, max_total_cites=MAX_TOTAL_CITES, w_sem=0.6
                        )
                    final_answers.append(ans_cited)

            # 4) 收集结果
            for (qid, doc_id, gold, pid_list), ctx, ans_cited in zip(batch_meta_store, ctx_list, final_answers):
                rows.append({
                    "qid": qid,
                    "doc_id": doc_id,
                    "question": batch_q_store[0] if False else None,  # 占位，下面单条回填
                    "gold_answer": gold,
                    "pid_list": pid_list,
                    "contexts": ctx.split("\n"),  # 可换成 [p['text'] for p in passages]
                    "answer_M3": ans_cited,
                    "topk_sha256": j_hash
                })

            # 回填真实 question 文本（避免重复大串复制影响显存，单独回填）
            for i in range(len(rows) - len(batch_q_store), len(rows)):
                rows[i]["question"] = batch_q_store[i - (len(rows) - len(batch_q_store))]

            # 清空批
            batch_pass_store.clear()
            batch_q_store.clear()
            batch_meta_store.clear()

# 处理尾批
if batch_q_store:
    ctx_list = build_ctx_frozen_batch(batch_pass_store)
    prompts  = make_prompts_from_ctx_questions(ctx_list, batch_q_store)
    gen_answers = generate_batch(model, tok, prompts)
    if FAST_NO_CITATION:
        final_answers = gen_answers
    else:
        final_answers = []
        for ans, passages in zip(gen_answers, batch_pass_store):
            if FAST_LEXICAL_CITE:
                saved_USE_E5 = USE_E5_SIM
                USE_E5_SIM = False
                ans_cited = attach_citations_sentwise(
                    ans, passages, per_sent_top=1, min_score_sent=MIN_CITE_SCORE,
                    global_fallback=True, max_total_cites=MAX_TOTAL_CITES, w_sem=0.0
                )
                USE_E5_SIM = saved_USE_E5
            else:
                ans_cited = attach_citations_sentwise(
                    ans, passages, per_sent_top=1, min_score_sent=MIN_CITE_SCORE,
                    global_fallback=True, max_total_cites=MAX_TOTAL_CITES, w_sem=0.6
                )
            final_answers.append(ans_cited)

    for (qid, doc_id, gold, pid_list), q_text, ctx, ans_cited in zip(batch_meta_store, batch_q_store, ctx_list, final_answers):
        rows.append({
            "qid": qid,
            "doc_id": doc_id,
            "question": q_text,
            "gold_answer": gold,
            "pid_list": pid_list,
            "contexts": ctx.split("\n"),
            "answer_M3": ans_cited,
            "topk_sha256": j_hash
        })

In [ ]:
# 保存
ensure_dir(str(Path(OUT_CSV).parent))
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
df.to_parquet(OUT_PARQUET, index=False)
print("✓ Saved TEST predictions (FROZEN, batched):")
print(" •", OUT_CSV)
print(" •", OUT_PARQUET)
try:
    from IPython.display import display
    display(df.head(3)[["qid","doc_id","question","answer_M3","contexts"]])
except Exception:
    print(df.head(3)[["qid","doc_id","question","answer_M3","contexts"]])